# P053 — Day 1 Production Training on Colab A100
## DRAM Memory Yield Predictor — HybridTransformerCNN

---

### What This Notebook Does

This notebook trains the **Day 1 champion model** — the first production model deployed to AWS.

**The production story:**
You're a principal ML engineer at a DRAM manufacturer (Micron, Samsung, SK Hynix). Your fab processes **50,000 wafers/month**, each worth **$1,200**. Every wafer has hundreds of memory dies, and your job is to predict which dies will fail **before** they ship to customers.

**The numbers that matter:**
- **16 million** test records for training
- **0.62%** fail rate (1:159 class imbalance — only 62K fails in 10M rows)
- **$45,000** per missed defect (shipped bad chip → customer recall)
- **$120** per false alarm (unnecessary re-test)

### Where This Fits in the 40-Day Production Lifecycle

```
┌─────────────────────────────────────────────────────────────────────────┐
│                          40-DAY PRODUCTION FLOW                         │
│                                                                         │
│  ┌─────────────┐   ┌───────────────────────────────────────────────┐   │
│  │ THIS NOTEBOOK│   │            AWS EC2 (Automated)                │   │
│  │ (Colab A100) │   │                                               │   │
│  │              │   │  Day 2-19: Daily inference → drift check ✅    │   │
│  │  Day 1:      │   │  Day 20:   Drift detected → retrain → v2 ✅   │   │
│  │  Train v1    │──→│  Day 21-30: Inference with v2                  │   │
│  │  champion    │   │  Day 31:   Severe drift → retrain → v3 ✅     │   │
│  │              │   │  Day 32-38: Inference with v3                  │   │
│  │  Upload to   │   │  Day 39:   Bad deploy → rollback → v4 ✅      │   │
│  │  Google Drive│   │  Day 40:   Final metrics logged               │   │
│  └─────────────┘   └───────────────────────────────────────────────┘   │
│                                                                         │
│  Airflow orchestrates | Kafka streams | Spark processes | MLflow tracks │
└─────────────────────────────────────────────────────────────────────────┘
```

**After this notebook completes:**
1. Model weights saved to Google Drive → downloaded to local → pushed to S3
2. Model registered in MLflow Model Registry as v1 @champion
3. Docker image pushed to ECR via CI/CD
4. 40-day simulation starts on AWS EC2 (Airflow + Kafka + Spark + MLflow)
5. Drift detection, retraining, rollback — all automated on AWS

### Model Architecture: HybridTransformerCNN

```
Input: 36 features (31 tabular + 3 spatial + 2 engineered)
  │
  ├─→ Tabular Branch ─→ [2-Layer Transformer (4 heads, d=128)] ─→ 128-dim
  │                       Captures feature interactions via self-attention
  │
  ├─→ Spatial Branch ──→ [Conv1D (3→32→64)] ────────────────────→ 64-dim
  │                       Captures die position patterns (edge effects)
  │
  └─→ Fusion MLP ─────→ [192→128→relu→dropout→64→1→sigmoid] ──→ fail probability
                          317,633 parameters total
```

**Why this architecture?**
- Tabular Transformer captures complex feature interactions (e.g., leakage × temperature)
- Spatial CNN captures die position effects (edge dies fail 3× more often)
- Fusion MLP learns which branch to trust for each prediction
- 317K params — small enough for <10ms inference latency

### Training Strategy

| Setting | Value | Why |
|---------|-------|-----|
| Loss | FocalLoss(α=0.75, γ=2.0) | Handles 1:159 imbalance — down-weights easy negatives |
| AMP | bfloat16 (A100) | 8-bit exponent = no overflow risk (float16 collapsed!) |
| LR | 1e-3 + CosineAnnealing | Aggressive start, smooth decay to 1e-6 |
| Batch | 4096 | A100 has 40 GB VRAM — utilize it |
| Patience | 12 epochs | Early stopping prevents overfitting |
| Oversampling | WeightedRandomSampler | Ensures 50/50 pos/neg in each batch |

---

**Prerequisites:**
- Google Colab Pro ($10/mo) with A100 GPU runtime
- Repository cloned from GitHub
- ~15 min to generate data from scratch (or load from Drive if cached)


## Step 1: Environment Setup

We install exact versions needed for reproducibility. This only runs on Colab.

**Key packages:**
- `mlflow>=2.15` — Model Registry with alias support (@champion, @challenger)
- `torch` — PyTorch with CUDA 12.x + bfloat16 Tensor Core support
- `psutil` — Hardware detection for reproducibility logging


In [4]:
# ━━━ Installation (Colab only) ━━━
import subprocess, sys

packages = [
    "mlflow>=2.15", "torch", "scikit-learn", "numpy", "pandas",
    "matplotlib", "psutil",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All packages installed")


✅ All packages installed


## Step 2: Mount Google Drive & Clone Repository

**Why Google Drive?** Colab VMs are ephemeral — when the session ends, your data dies. Drive persists across sessions. After training, we save:
- Model weights (.pt file, ~1.2 MB)
- Benchmark JSON (training metrics, hardware info)
- MLflow SQLite DB (experiment history)

Later, we download these to local and push to AWS S3 for the production deployment.

**Why clone from GitHub?** This gives us access to all `src/` modules:
- `config.py` — Central config (model architecture, hyperparameters, business metrics)
- `data_generator.py` — Synthetic DRAM data with temporal drift injection
- `model.py` — HybridTransformerCNN + dataloaders with oversampling
- `focal_loss.py` — FocalLoss with label smoothing (handles 1:159 imbalance)
- `mlflow_utils.py` — MLflow tracking, logging, Model Registry helpers


In [5]:
# ━━━ Google Drive Mount & Repository Clone ━━━
import os, sys, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/AIML-Engineering-Lab/053_dram_memory_yield_mlops.git"
PROJECT_DIR = Path("/content/053_memory_yield_predictor")

if PROJECT_DIR.exists():
    os.chdir(PROJECT_DIR)
    os.system("git pull origin main")
    print("✅ Repository updated")
else:
    os.system(f"git clone {REPO_URL} {PROJECT_DIR}")
    os.chdir(PROJECT_DIR)
    print("✅ Repository cloned")

sys.path.insert(0, str(PROJECT_DIR))
print(f"📂 Working directory: {os.getcwd()}")
print(f"📦 Modules: {sorted([f.stem for f in (PROJECT_DIR / 'src').glob('*.py') if not f.stem.startswith('_')])}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Repository updated
📂 Working directory: /content/053_memory_yield_predictor
📦 Modules: ['benchmark_mps', 'compare_models', 'config', 'data_generator', 'data_profile', 'drift_detector', 'eda_plots', 'focal_loss', 'inference', 'kafka_consumer', 'kafka_producer', 'load_test', 'mlflow_utils', 'model', 'pandas_vs_spark_benchmark', 'plot_simulation_results', 'preprocess', 'retrain_trigger', 'retrolog_experiments', 'run_simulation', 'sagemaker_pipeline', 'serve', 'spark_drift_detector', 'spark_etl', 'streaming_data_generator', 'train', 'train_baseline']


## Step 3: Hardware Detection — Critical for AMP Strategy

This is not just "checking GPU name." The hardware detection determines **how we train**:

| GPU | Compute Capability | AMP Strategy | GradScaler | Why |
|-----|-------------------|-------------|------------|-----|
| **A100** | 8.0 | **bfloat16** | ❌ Disabled | 8-bit exponent = same range as float32. No overflow. |
| T4/V100 | 7.5/7.0 | float16 | ✅ Enabled | 5-bit exponent = max 65,504. Overflow risk with FocalLoss. |
| CPU | N/A | float32 | ❌ N/A | Full precision, 100× slower. Don't do this. |

### The bfloat16 Story (This Is Interview Gold)

Our A100 v2 training (float16) **collapsed at epoch 5**:
- FocalLoss with 1:160 imbalance produces gradients > 65,504 at LR=1e-3
- float16 max value = 65,504 (5-bit exponent) → overflow → inf → NaN
- GradScaler detects inf, halves scale repeatedly → scale reaches 0 → all-zero gradients → model is dead

v3 added warmup (5 epochs) — collapsed at epoch 7 instead. Warmup delayed the problem, didn't fix it.

**The fix:** bfloat16 has an 8-bit exponent (range up to ~3.4×10³⁸, same as float32). No overflow possible. No GradScaler needed. Training stable to 50+ epochs.

We also enable **TF32** on A100 — uses Tensor Cores for ~3× faster matrix multiply with minimal precision loss.

> **Interview question:** *"How do you decide between float16 and bfloat16?"*
> If GPU compute capability ≥ 8.0 (Ampere+), always bfloat16. For older GPUs, float16 with GradScaler and lower LR (<3e-4).


In [6]:
# ━━━ Hardware Detection ━━━
import torch
import psutil

def detect_hardware():
    """Auto-detect GPU and return optimal training settings."""
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        cc = torch.cuda.get_device_capability(0)

        if cc[0] >= 8:  # A100/H100/L4 (Ampere+)
            amp_dtype = "bfloat16"
            use_amp = True
            use_gradscaler = False
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        else:  # T4/V100
            amp_dtype = "float16"
            use_amp = True
            use_gradscaler = True

        device = torch.device("cuda")
    else:
        gpu_name, vram_gb = "CPU", 0
        amp_dtype, use_amp, use_gradscaler = "float32", False, False
        device = torch.device("cpu")

    return {
        "device": device, "gpu_name": gpu_name, "vram_gb": round(vram_gb, 1),
        "amp_dtype": amp_dtype, "use_amp": use_amp, "use_gradscaler": use_gradscaler,
    }

hw = detect_hardware()
print(f"{'='*60}")
print(f"  GPU:           {hw['gpu_name']}")
print(f"  VRAM:          {hw['vram_gb']} GB")
print(f"  AMP dtype:     {hw['amp_dtype']}")
print(f"  GradScaler:    {hw['use_gradscaler']}")
print(f"  TF32:          {torch.backends.cuda.matmul.allow_tf32 if torch.cuda.is_available() else 'N/A'}")
print(f"  RAM:           {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"  CPU cores:     {psutil.cpu_count()}")
print(f"{'='*60}")

if hw['gpu_name'] == 'CPU':
    print("⚠️  No GPU! Go to Runtime → Change runtime type → GPU → A100")


  GPU:           NVIDIA A100-SXM4-40GB
  VRAM:          42.4 GB
  AMP dtype:     bfloat16
  GradScaler:    False
  TF32:          True
  RAM:           89.6 GB
  CPU cores:     12


## Step 4: Load the Preprocessed Dataset

The data pipeline:

```
src/data_generator.py            → 16M raw DRAM test records
  ├─ 10M train + 2M val + 2M test + 2M unseen (6-month drift offset)
  ├─ 23 electrical test parameters + 4 categorical + 3 spatial
  └─ Temporal drift, spatial edge effects, 10 data quality issues

src/preprocess.py                → Preprocessed arrays
  ├─ Impute (KNN) → Winsorize (IQR×3) → Log-transform (skewed features)
  ├─ Engineer 7 features (retention×temp, leakage/retention ratio, etc.)
  ├─ Encode categoricals (ordinal) → StandardScaler → float32
  └─ Output: preprocessed_full.npz (2.1 GB, DVC-tracked)
```

**Why NPZ instead of raw parquet?**
Loading the pre-processed NPZ takes ~10s. Regenerating from raw + full preprocessing takes ~15 min.
The NPZ is generated once locally, uploaded to Drive, and reused across Colab sessions.

**Feature engineering creates 7 physics-based features:**
- `retention_temp_interaction` — retention × temperature (thermal sensitivity)
- `leakage_retention_ratio` — leakage ÷ retention (degradation pattern)
- `voltage_stress_index` — composite stress metric
- `spatial_risk_score` — edge distance + neighbor defects
- And 3 more interaction terms capturing physics failure modes


In [7]:
# ━━━ Load Preprocessed Dataset ━━━
import numpy as np

DRIVE_DATA = Path("/content/drive/MyDrive/P053_artifacts/data")
LOCAL_DATA = PROJECT_DIR / "data"
LOCAL_DATA.mkdir(exist_ok=True)

preprocessed_path = LOCAL_DATA / "preprocessed_full.npz"

if (DRIVE_DATA / "preprocessed_full.npz").exists():
    import shutil
    shutil.copy(DRIVE_DATA / "preprocessed_full.npz", preprocessed_path)
    print(f"✅ Loaded from Drive ({preprocessed_path.stat().st_size / 1e9:.1f} GB)")

elif preprocessed_path.exists():
    print(f"✅ Already exists locally ({preprocessed_path.stat().st_size / 1e9:.1f} GB)")

else:
    print("⏳ Generating full 16M-row dataset from scratch (~15 min)...")
    t0 = time.time()
    from src.data_generator import generate_all_splits
    generate_all_splits()
    print(f"   Data generation: {time.time() - t0:.1f}s")

    print("⏳ Preprocessing: impute → winsorize → log → engineer → scale...")
    t0 = time.time()
    from src.preprocess import preprocess_pipeline
    preprocess_pipeline(use_full=True)
    print(f"   Preprocessing: {time.time() - t0:.1f}s")

    # Cache to Drive
    DRIVE_DATA.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy(preprocessed_path, DRIVE_DATA / "preprocessed_full.npz")
    print("✅ Cached to Drive for future sessions")

data = dict(np.load(preprocessed_path, allow_pickle=True))
feature_names = list(data['feature_names'])

print(f"\n{'='*60}")
print(f"  DATASET SUMMARY")
print(f"{'='*60}")
for split in ['train', 'val', 'test', 'unseen']:
    n = data[f'X_{split}'].shape[0]
    fails = int(data[f'y_{split}'].sum())
    rate = data[f'y_{split}'].mean() * 100
    print(f"  {split:>7}: {n:>12,} rows | {fails:>6,} fails ({rate:.2f}%)")
print(f"  {'total':>7}: {sum(data[f'X_{s}'].shape[0] for s in ['train','val','test','unseen']):>12,} rows")
print(f"{'='*60}")
print(f"  Features: {len(feature_names)} → {feature_names[:5]}...")
print(f"  Imbalance: 1:{int(1/data['y_train'].mean()):.0f}")
print(f"{'='*60}")


⏳ Generating full 16M-row dataset from scratch (~15 min)...

  Split: train
  Generating 10,000,000 samples (split=train, seed=42)...
    Rows: 10,000,000 | Fails: 54,632 (0.55%) | Label noise: 12,464 | Missing: 3,796,537 cells | Time: 83.2s
    Saved: dram_stdf_train.parquet (537.0 MB)

  Split: val
  Generating 2,000,000 samples (split=val, seed=123)...
    Rows: 2,000,000 | Fails: 14,422 (0.72%) | Label noise: 2,687 | Missing: 760,106 cells | Time: 14.6s
    Saved: dram_stdf_val.parquet (107.9 MB)

  Split: test
  Generating 2,000,000 samples (split=test, seed=456)...
    Rows: 2,000,000 | Fails: 11,700 (0.58%) | Label noise: 2,567 | Missing: 760,174 cells | Time: 14.7s
    Saved: dram_stdf_test.parquet (107.8 MB)

  Split: unseen
  Generating 2,000,000 samples (split=unseen, seed=999)...
    Rows: 2,000,000 | Fails: 15,516 (0.78%) | Label noise: 2,791 | Missing: 759,464 cells | Time: 14.3s
    Saved: dram_stdf_unseen.parquet (108.2 MB)

  Stats saved to data_generation_stats.json
 

## Step 5: Initialize MLflow Tracking

**Why local SQLite on Colab instead of AWS RDS?**

1. **No network dependency** — Colab GPU sessions are precious ($10/mo). Training shouldn't fail because of a flaky network connection to AWS RDS.
2. **Speed** — Local writes are instant. Remote HTTP API adds latency per epoch.
3. **Portability** — The `.db` file gets saved to Drive, downloaded locally, then imported into Docker PostgreSQL MLflow.

**In production (AWS):** MLflow points directly to RDS PostgreSQL + S3 artifacts. The Airflow retrain DAG writes there natively.

**Transfer path:** Colab SQLite → Drive → local → Docker PostgreSQL → AWS RDS


In [8]:
# ━━━ MLflow Setup ━━━
import mlflow

os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{PROJECT_DIR / 'mlflow_colab.db'}"

from src.mlflow_utils import init_mlflow
experiment_id = init_mlflow("P053-Production-Training")

print(f"✅ MLflow initialized")
print(f"  Experiment: {experiment_id}")
print(f"  Tracking:   {mlflow.get_tracking_uri()}")


2026/04/06 05:18:09 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/06 05:18:09 INFO mlflow.store.db.utils: Updating database tables


✅ MLflow initialized
  Experiment: 1
  Tracking:   sqlite:////content/053_memory_yield_predictor/mlflow_colab.db


## Step 6: Define Training Functions

Three core functions:

### `train_one_epoch()` — Forward + backward pass
- `torch.autocast()` for mixed precision (bfloat16 on A100)
- Gradient clipping to `max_norm=1.0` (prevents FocalLoss gradient explosions)
- Samples AUC-PR every 10th batch (primary metric, not accuracy)

### `evaluate_split()` — Eval without gradients
- Batched inference on 2M-row splits (avoids OOM)
- Returns loss, AUC-PR, and raw probabilities for threshold optimization

### `run_training_session()` — The main loop
1. Creates dataloaders with WeightedRandomSampler (50/50 pos/neg per batch)
2. Builds HybridTransformerCNN (or loads pretrained weights for fine-tuning)
3. FocalLoss(α=0.75, γ=2.0) — focuses on hard-to-classify minority examples
4. CosineAnnealing LR scheduler (smooth decay from 1e-3 to 1e-6)
5. Early stopping with patience=12
6. Evaluates on val/test/unseen splits
7. Finds optimal classification threshold (F1-maximizing on validation)
8. Logs everything to MLflow (params, per-epoch metrics, model artifact)
9. Saves benchmark JSON (for later import to Docker PostgreSQL MLflow)

**Why FocalLoss over BCE?** At 1:159 imbalance, BCE lets the model cheat — predict "pass" on everything for 99.4% accuracy. FocalLoss with γ=2.0 down-weights easy negatives so the model must actually learn to find the rare failures.


In [9]:
# ━━━ Training Functions ━━━
import numpy as np
import torch
from sklearn.metrics import (
    average_precision_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_auc_score,
)

from src.config import MODEL_PARAMS, TRAINING
from src.model import HybridTransformerCNN, create_dataloaders, find_best_threshold
from src.focal_loss import FocalLossWithLabelSmoothing
from src.mlflow_utils import (
    start_training_run, log_epoch_metrics, log_evaluation_results,
    log_model_artifact, log_training_summary,
)


def train_one_epoch(model, loader, criterion, optimizer, device, scaler, hw_info):
    model.train()
    total_loss, n_batches = 0, 0
    sample_preds, sample_labels = [], []

    amp_ctx = (
        torch.autocast("cuda", dtype=torch.bfloat16 if hw_info["amp_dtype"] == "bfloat16" else torch.float16)
        if hw_info["use_amp"]
        else torch.autocast("cpu", enabled=False)
    )

    for batch_idx, batch in enumerate(loader):
        x_tab, x_spa, labels = [b.to(device) for b in batch]
        optimizer.zero_grad(set_to_none=True)

        with amp_ctx:
            logits = model(x_tab, x_spa)
            loss = criterion(logits, labels)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item()
        n_batches += 1

        if batch_idx % 10 == 0:
            with torch.no_grad():
                preds = torch.sigmoid(logits).cpu().numpy()
                sample_preds.extend(preds)
                sample_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / n_batches
    auc_pr = average_precision_score(
        np.array(sample_labels), np.array(sample_preds)
    ) if np.array(sample_labels).sum() > 0 else 0
    return avg_loss, auc_pr


@torch.no_grad()
def evaluate_split(model, X, y, feature_names, device, criterion, hw_info, batch_size=2048):
    model.eval()
    spatial_cols = ["die_x", "die_y", "edge_distance"]
    spatial_idx = [feature_names.index(c) for c in spatial_cols if c in feature_names]
    tabular_idx = [i for i in range(len(feature_names)) if i not in spatial_idx]

    X_tab = torch.tensor(X[:, tabular_idx].astype(np.float32))
    X_spa = torch.tensor(X[:, spatial_idx].astype(np.float32))

    amp_ctx = (
        torch.autocast("cuda", dtype=torch.bfloat16 if hw_info["amp_dtype"] == "bfloat16" else torch.float16)
        if hw_info["use_amp"]
        else torch.autocast("cpu", enabled=False)
    )

    all_logits, total_loss, n_batches = [], 0, 0
    for i in range(0, len(X_tab), batch_size):
        xt = X_tab[i:i+batch_size].to(device)
        xs = X_spa[i:i+batch_size].to(device)
        lb = torch.tensor(y[i:i+batch_size].astype(np.float32)).to(device)
        with amp_ctx:
            logits = model(xt, xs)
            loss = criterion(logits, lb)
        all_logits.append(logits.cpu())
        total_loss += loss.item()
        n_batches += 1

    logits = torch.cat(all_logits)
    proba = torch.sigmoid(logits).numpy()
    avg_loss = total_loss / n_batches
    auc_pr = average_precision_score(y, proba) if y.sum() > 0 else 0
    return avg_loss, auc_pr, proba


def run_training_session(
    session_name, run_name, data, hw_info,
    epochs=50, lr=1e-3, batch_size=4096, patience=12,
    model_weights_path=None
):
    device = hw_info["device"]
    t_global = time.time()

    print(f"\n{'='*70}")
    print(f"  {session_name}")
    print(f"  GPU: {hw_info['gpu_name']} | AMP: {hw_info['amp_dtype']} | LR: {lr} | Batch: {batch_size}")
    if model_weights_path:
        print(f"  Fine-tuning from: {Path(model_weights_path).name}")
    else:
        print(f"  Training from scratch (random initialization)")
    print(f"{'='*70}\n")

    X_train, y_train = data["X_train"], data["y_train"]
    X_val, y_val = data["X_val"], data["y_val"]
    X_test, y_test = data["X_test"], data["y_test"]
    X_unseen, y_unseen = data["X_unseen"], data["y_unseen"]
    feature_names = list(data["feature_names"])

    train_loader, val_loader, n_tab, n_spa = create_dataloaders(
        X_train, y_train, X_val, y_val, feature_names,
        batch_size=batch_size, oversample=True,
    )

    model = HybridTransformerCNN(
        n_tabular=n_tab, n_spatial=n_spa,
        d_model=MODEL_PARAMS["d_model"],
        n_heads=MODEL_PARAMS["n_heads"],
        n_layers=MODEL_PARAMS["n_layers"],
        cnn_out=MODEL_PARAMS["cnn_out"],
        dropout=MODEL_PARAMS["dropout"],
    ).to(device)

    if model_weights_path and Path(model_weights_path).exists():
        model.load_state_dict(torch.load(model_weights_path, weights_only=True, map_location=device))
        print(f"  ✅ Loaded pretrained weights")

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Model: {n_params:,} parameters")

    criterion = FocalLossWithLabelSmoothing(
        alpha=TRAINING["focal_alpha"], gamma=TRAINING["focal_gamma"],
        smoothing=TRAINING["label_smoothing"],
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=TRAINING["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if hw_info["use_gradscaler"] else None

    run_ctx = start_training_run(
        run_name=run_name, gpu_name=hw_info["gpu_name"], amp_dtype=hw_info["amp_dtype"],
        batch_size=batch_size, learning_rate=lr,
        extra_params={
            "hw.vram_gb": hw_info["vram_gb"],
            "train.rows": int(len(y_train)),
            "train.patience": patience,
            "session": session_name,
        },
        extra_tags={"gpu_type": hw_info["gpu_name"], "run_context": "colab-pro"},
    )

    with run_ctx as run:
        print(f"  MLflow run: {run.info.run_id}\n")

        history = {"train_loss": [], "val_loss": [], "train_auc_pr": [], "val_auc_pr": []}
        epoch_times = []
        best_val_auc, best_epoch, patience_counter = 0, 0, 0
        save_path = PROJECT_DIR / "src" / "artifacts" / f"hybrid_best_{run_name}.pt"
        save_path.parent.mkdir(parents=True, exist_ok=True)

        print(f"{'Ep':>4} {'TrLoss':>8} {'VaLoss':>8} {'TrAUC':>8} {'VaAUC':>8} {'LR':>10} {'Time':>7} {'Samp/s':>10}")
        print("-" * 78)

        for epoch in range(1, epochs + 1):
            t_epoch = time.time()

            train_loss, train_auc = train_one_epoch(
                model, train_loader, criterion, optimizer, device, scaler, hw_info
            )
            val_loss, val_auc, _ = evaluate_split(
                model, X_val, y_val, feature_names, device, criterion, hw_info
            )

            scheduler.step()
            epoch_time = time.time() - t_epoch
            epoch_times.append(epoch_time)
            throughput = len(y_train) / epoch_time

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["train_auc_pr"].append(train_auc)
            history["val_auc_pr"].append(val_auc)

            current_lr = optimizer.param_groups[0]["lr"]
            log_epoch_metrics(epoch, train_loss, val_loss, train_auc, val_auc, current_lr, epoch_time, throughput)

            improved = ""
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_epoch = epoch
                patience_counter = 0
                improved = " ★"
                torch.save(model.state_dict(), save_path)
            else:
                patience_counter += 1

            if epoch <= 5 or epoch % 5 == 0 or improved:
                print(f"{epoch:>4} {train_loss:>8.4f} {val_loss:>8.4f} {train_auc:>8.4f} {val_auc:>8.4f} {current_lr:>10.6f} {epoch_time:>6.1f}s {throughput:>9,.0f}{improved}")

            if patience_counter >= patience:
                print(f"\n⏹️  Early stopping at epoch {epoch} (best={best_epoch})")
                break

        # ── Evaluation ──
        model.load_state_dict(torch.load(save_path, weights_only=True, map_location=device))

        print(f"\n{'='*70}")
        print(f"  EVALUATION — best model from epoch {best_epoch}")
        print(f"{'='*70}")

        results = {}
        threshold = None
        for split_name, X_s, y_s in [("val", X_val, y_val), ("test", X_test, y_test), ("unseen", X_unseen, y_unseen)]:
            _, split_auc, proba = evaluate_split(model, X_s, y_s, feature_names, device, criterion, hw_info)

            if split_name == "val":
                threshold = find_best_threshold(y_s, proba)

            y_pred = (proba >= threshold).astype(int)
            cm = confusion_matrix(y_s, y_pred)
            tn, fp, fn, tp = cm.ravel()

            results[split_name] = {
                "precision": float(precision_score(y_s, y_pred, zero_division=0)),
                "recall": float(recall_score(y_s, y_pred, zero_division=0)),
                "f1": float(f1_score(y_s, y_pred, zero_division=0)),
                "auc_pr": float(split_auc),
                "auc_roc": float(roc_auc_score(y_s, proba)),
                "threshold": float(threshold),
                "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
            }
            print(f"  {split_name.upper():>7}: AUC-PR={split_auc:.4f} | F1={results[split_name]['f1']:.4f} | Recall={results[split_name]['recall']:.4f} | TP={tp:,} FP={fp:,} FN={fn:,}")

        log_evaluation_results(results, threshold)

        total_time = time.time() - t_global
        peak_vram = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

        log_training_summary(
            best_epoch=best_epoch, best_val_auc=best_val_auc,
            total_time_min=total_time / 60, avg_epoch_time_s=np.mean(epoch_times),
            throughput=len(y_train) / min(epoch_times) if epoch_times else 0,
            peak_vram_gb=peak_vram, epochs_run=len(epoch_times), train_rows=len(y_train),
        )

        if save_path.exists():
            log_model_artifact(save_path)

        import json
        benchmark = {
            "session": session_name, "run_name": run_name,
            "device": str(device), "gpu_name": hw_info["gpu_name"],
            "gpu_vram_gb": hw_info["vram_gb"], "amp_dtype": hw_info["amp_dtype"],
            "batch_size": batch_size, "lr": lr, "model_params": n_params,
            "train_rows": len(y_train), "epochs_run": len(epoch_times),
            "best_epoch": best_epoch, "best_val_auc": best_val_auc,
            "avg_epoch_time_s": float(np.mean(epoch_times)),
            "total_train_time_min": float(total_time / 60),
            "peak_gpu_memory_gb": float(peak_vram),
            "results": results, "history": history,
            "mlflow_run_id": run.info.run_id,
        }
        benchmark_path = PROJECT_DIR / "data" / f"benchmark_{run_name}.json"
        with open(benchmark_path, "w") as f:
            json.dump(benchmark, f, indent=2)
        mlflow.log_artifact(str(benchmark_path), "benchmark")

        print(f"\n🏁 Training complete in {total_time/60:.1f} min")
        print(f"   Best Val AUC-PR: {best_val_auc:.4f} (epoch {best_epoch})")
        print(f"   Peak VRAM: {peak_vram:.1f} GB | Throughput: {len(y_train)/np.mean(epoch_times):,.0f} samples/sec")

    return {
        "run_id": run.info.run_id, "best_epoch": best_epoch,
        "best_val_auc": best_val_auc, "results": results,
        "save_path": str(save_path), "benchmark": benchmark,
    }

print(f"✅ Training functions defined")
print(f"   Architecture: HybridTransformerCNN ({MODEL_PARAMS['d_model']}d, {MODEL_PARAMS['n_heads']}h, {MODEL_PARAMS['n_layers']}L)")
print(f"   Loss: FocalLoss(α={TRAINING['focal_alpha']}, γ={TRAINING['focal_gamma']})")


✅ Training functions defined
   Architecture: HybridTransformerCNN (128d, 4h, 2L)
   Loss: FocalLoss(α=0.75, γ=2.0)


---

## 🏋️ Day 1 — Train the Champion Model

This is the moment. 10 million rows, A100 GPU, bfloat16 precision, ~85 minutes.

**Expected results:**
- AUC-PR ~0.05 on validation (10× better than random = 0.006)
- Early stopping around epoch 40-48
- Throughput ~80K-90K samples/sec on A100 (vs ~18K on T4)
- Peak VRAM ~12 GB of 40 GB

**What to watch for:**
- Val AUC-PR should increase steadily for 30+ epochs
- Train loss < Val loss = good (no overfitting)
- If val AUC-PR plateaus early (epoch 15), the model may need more capacity
- If train AUC-PR >> val AUC-PR, we're overfitting (increase dropout)

This model becomes **v1 @champion** — deployed to AWS for daily inference.


In [10]:
# ━━━ Day 1 — Train the Champion Model ━━━
result = run_training_session(
    session_name="Day 1 — Initial Production Training",
    run_name="A100-day1-champion",
    data=data,
    hw_info=hw,
    epochs=50,
    lr=1e-3,
    batch_size=4096,
    patience=12,
)

print(f"\n📊 Day 1 Results:")
for split, m in result["results"].items():
    print(f"  {split:>7}: AUC-PR={m['auc_pr']:.4f} | Recall={m['recall']:.4f} | F1={m['f1']:.4f}")



  Day 1 — Initial Production Training
  GPU: NVIDIA A100-SXM4-40GB | AMP: bfloat16 | LR: 0.001 | Batch: 4096
  Training from scratch (random initialization)

  Model: 317,633 parameters
  MLflow run: 036376de2ef7498ca2fee6a2fd71f5e8

  Ep   TrLoss   VaLoss    TrAUC    VaAUC         LR    Time     Samp/s
------------------------------------------------------------------------------


TypeError: Got unsupported ScalarType BFloat16

### Day 1 Analysis

Things to check in the output above:

1. **Convergence:** Did val AUC-PR improve through epoch 30+? If it plateaued at epoch 15, consider: more capacity, lower LR, different loss.

2. **Overfitting:** Is train AUC-PR >> val AUC-PR by more than 0.01? If yes, increase dropout from 0.2 → 0.3.

3. **Unseen split:** This simulates 6 months of drift. If unseen AUC-PR is much lower than test AUC-PR, the model is brittle — exactly what the 40-day simulation will stress test.

4. **Throughput:** A100 should hit 80K-90K samples/sec. If it's closer to 20K, check that bfloat16 is being used (not float32).

### What This Model Does in Production

After this notebook:
1. Model weights → S3 (`s3://p053-mlflow-artifacts/models/hybrid_best_A100-day1-champion.pt`)
2. Registered in MLflow Model Registry as v1 with alias **@champion**
3. Docker image (FastAPI + model) → ECR → EC2 deployment
4. **Every day for 40 days**, Airflow triggers:
   - Data generation (5M rows via `streaming_data_generator.py`)
   - Kafka → Spark ETL → Drift detection
   - Inference using THIS model
   - When drift is detected → retrain ON AWS → new model version


In [ ]:
# ━━━ Training Curves Visualization ━━━
import matplotlib.pyplot as plt

h = result["benchmark"]["history"]
epochs = range(1, len(h["val_auc_pr"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"P053 Day 1 — {hw['gpu_name']} {hw['amp_dtype']} Training", fontsize=14, fontweight='bold')

# Loss curves
axes[0].plot(epochs, h["train_loss"], label="Train", color="#2563EB", alpha=0.8)
axes[0].plot(epochs, h["val_loss"], label="Val", color="#DC2626", alpha=0.8, linestyle="--")
axes[0].axvline(result["best_epoch"], color="#059669", linestyle=":", alpha=0.5, label=f"Best (ep {result['best_epoch']})")
axes[0].set_title("Focal Loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# AUC-PR curves
axes[1].plot(epochs, h["train_auc_pr"], label="Train AUC-PR", color="#2563EB", alpha=0.8)
axes[1].plot(epochs, h["val_auc_pr"], label="Val AUC-PR", color="#DC2626", alpha=0.8, linestyle="--")
axes[1].axhline(0.006, color="gray", linestyle=":", alpha=0.3, label="Random baseline")
axes[1].axvline(result["best_epoch"], color="#059669", linestyle=":", alpha=0.5)
axes[1].set_title("AUC-PR (Primary Metric)"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUC-PR")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Final bar chart
splits = list(result["results"].keys())
auc_prs = [result["results"][s]["auc_pr"] for s in splits]
colors = ["#2563EB", "#059669", "#F59E0B"]
axes[2].bar(splits, auc_prs, color=colors)
axes[2].set_title("Final AUC-PR by Split"); axes[2].set_ylabel("AUC-PR")
axes[2].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(auc_prs):
    axes[2].text(i, v + 0.001, f"{v:.4f}", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(str(PROJECT_DIR / "assets" / "nb03_day1_training.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved to assets/nb03_day1_training.png")


## 💰 Business Impact Calculation

A principal engineer translates metrics into dollars. This is the ROI case for the model.


In [ ]:
# ━━━ Business Impact ━━━
from src.config import BUSINESS

wafers_per_month = BUSINESS["wafers_per_month"]       # 50,000
cost_fn = BUSINESS.get("cost_per_false_negative_usd", 45_000)
cost_fp = BUSINESS.get("cost_per_false_positive_usd", 120)
fail_rate = 0.0062

defective = wafers_per_month * fail_rate  # 310/month
test = result["results"]["test"]
caught = defective * test["recall"]
saved_fn = caught * cost_fn * 12  # Annual
cost_false_alarms = test["fp"] / (data["y_test"].shape[0] / wafers_per_month) * cost_fp * 12
net = saved_fn - cost_false_alarms

print(f"{'='*60}")
print(f"  BUSINESS IMPACT — Day 1 Champion Model")
print(f"{'='*60}")
print(f"  Wafers/month:         {wafers_per_month:>10,}")
print(f"  Fail rate:            {fail_rate*100:>10.2f}%")
print(f"  Defective/month:      {defective:>10,.0f}")
print(f"  Model recall:         {test['recall']:>10.4f}")
print(f"  Defects caught/month: {caught:>10,.0f}")
print(f"  Annual savings:       ${net:>10,.0f}")
print(f"  Per 1% recall gain:   ${defective * 0.01 * cost_fn * 12:>10,.0f}/year")
print(f"{'='*60}")
print(f"\n  Confusion matrix (test set):")
print(f"    TP={test['tp']:,} (correctly caught failures)")
print(f"    FP={test['fp']:,} (false alarms — $120 each)")
print(f"    FN={test['fn']:,} (missed failures — $45,000 each)")
print(f"    TN={test['tn']:,} (correctly passed)")


## 💾 Save Artifacts to Google Drive

Everything trained on Colab must persist on Drive. The transfer path to production:

```
Colab (this notebook)
  → Google Drive (persistent storage)
    → Download to Mac (local development)
      → Push to S3 (aws s3 cp)
        → EC2 pulls model for deployment
```

**Artifacts saved:**
- `hybrid_best_A100-day1-champion.pt` — Model weights (1.2 MB)
- `benchmark_A100-day1-champion.json` — All metrics, hyperparams, hardware info
- `mlflow_colab.db` — SQLite tracking DB (import to Docker PostgreSQL later)
- `nb03_day1_training.png` — Training curves plot


In [ ]:
# ━━━ Save All Artifacts to Google Drive ━━━
import shutil, json

DRIVE = Path("/content/drive/MyDrive/P053_artifacts")
for d in [DRIVE / "models", DRIVE / "benchmarks", DRIVE / "mlflow"]:
    d.mkdir(parents=True, exist_ok=True)

# Model weights
src = Path(result["save_path"])
if src.exists():
    shutil.copy(src, DRIVE / "models" / src.name)
    print(f"✅ Model → {src.name} ({src.stat().st_size / 1e6:.1f} MB)")

# Benchmark JSON
bm_path = DRIVE / "benchmarks" / f"benchmark_{result['benchmark']['run_name']}.json"
with open(bm_path, "w") as f:
    json.dump(result["benchmark"], f, indent=2)
print(f"✅ Benchmark → {bm_path.name}")

# MLflow DB
mlflow_db = PROJECT_DIR / "mlflow_colab.db"
if mlflow_db.exists():
    shutil.copy(mlflow_db, DRIVE / "mlflow" / "mlflow_colab.db")
    print(f"✅ MLflow DB → mlflow_colab.db ({mlflow_db.stat().st_size / 1e6:.1f} MB)")

# Training plot
plot = PROJECT_DIR / "assets" / "nb03_day1_training.png"
if plot.exists():
    shutil.copy(plot, DRIVE / "nb03_day1_training.png")
    print(f"✅ Plot → nb03_day1_training.png")

print(f"\n🎯 All artifacts saved to: {DRIVE}")


---

## 🏁 Day 1 Training Complete — Next Steps

### Transfer to Production (do on your Mac)

```bash
# 1. Download from Google Drive to local
cp ~/Google\ Drive/My\ Drive/P053_artifacts/models/*.pt \
   ~/AIML/55_LinkedIn/053_memory_yield_predictor/src/artifacts/

# 2. Push model to S3
aws s3 cp src/artifacts/hybrid_best_A100-day1-champion.pt \
   s3://p053-mlflow-artifacts/models/v1/

# 3. Import benchmark into Docker PostgreSQL MLflow
python src/retrolog_experiments.py --benchmark data/benchmark_A100-day1-champion.json

# 4. Register in Model Registry
python -c "
from src.mlflow_utils import register_model
register_model(run_id='<RUN_ID>', version_description='Day 1 A100 champion', alias='champion')
"

# 5. Tag Docker image and push to ECR (triggers CI/CD)
git tag v1.0.0
git push origin v1.0.0

# 6. Deploy on EC2 and start 40-day simulation
ssh -i ~/.ssh/p053-key.pem ec2-user@<EC2_IP>
cd 053_dram_memory_yield_mlops
docker-compose -f deploy/docker-compose-bigdata.yml up -d
# Airflow UI: http://<EC2_IP>:8080 — trigger dag_simulation_master
```

### What Happens on AWS (40 Days)

| Day | Event | Airflow Action |
|-----|-------|----------------|
| 1 | Model v1 deployed | Start daily pipeline DAG |
| 2-19 | Daily: 5M rows → Kafka → Spark → inference → drift check | All pass ✅ |
| 20 | PSI > 0.10 on 4 features | Retrain DAG → v2 → canary → promote |
| 21-30 | Daily inference with v2 | Drift stabilized |
| 31 | PSI > 0.20 on 7 features | Retrain DAG → v3 → canary → promote |
| 32-38 | Daily inference with v3 | Monitoring |
| 39 | Canary deliberately fails | Rollback to v3, from-scratch → v4 |
| 40 | Final metrics logged | Simulation complete |

**Total data processed:** 200M rows (5M/day × 40 days)
**Infrastructure:** Kafka + Spark + Airflow + MLflow + Prometheus + Grafana — all on EC2
**Cost:** ~$6 AWS total

### Interview Stories This Generates

1. *"Train first model on Colab A100 — bfloat16, 16M rows, 85 min"*
2. *"Deploy to AWS via CI/CD — ECR, docker-compose, FastAPI"*
3. *"40-day automated simulation — Airflow orchestrates Kafka + Spark + drift detection"*
4. *"3 retrain events, 1 rollback — all automated, no manual intervention"*
5. *"Total cost: $10 Colab + $6 AWS = $16 for a production-grade ML lifecycle"*


In [ ]:
# ━━━ Final Summary ━━━
b = result["benchmark"]
print(f"{'='*70}")
print(f"  P053 — DAY 1 TRAINING COMPLETE")
print(f"{'='*70}")
print(f"  GPU:              {hw['gpu_name']}")
print(f"  AMP:              {hw['amp_dtype']}")
print(f"  Total time:       {b['total_train_time_min']:.1f} min")
print(f"  Epochs:           {b['epochs_run']} (best: {b['best_epoch']})")
print(f"  Val AUC-PR:       {b['best_val_auc']:.4f}")
print(f"  Test AUC-PR:      {result['results']['test']['auc_pr']:.4f}")
print(f"  Unseen AUC-PR:    {result['results']['unseen']['auc_pr']:.4f}")
print(f"  Peak VRAM:        {b['peak_gpu_memory_gb']:.1f} GB")
print(f"  Throughput:       {int(b['train_rows'] / b['avg_epoch_time_s']):,} samples/sec")
print(f"  MLflow run:       {result['run_id']}")
print(f"  Model saved:      {Path(result['save_path']).name}")
print(f"{'='*70}")
print(f"\n  → Next: Transfer to AWS and start 40-day simulation")
print(f"  → See: docs/AWS_SETUP_GUIDE.md for detailed instructions")
